# Módulo 5 — Conexão com Snowflake

Neste notebook vamos entender a classe de conexão usada no projeto de qualidade de dados
da Equatorial e aprender a usá-la para executar queries e carregar dados em DataFrames.

## O que vamos ver

1. Estrutura do módulo `snowflake_db.py`
2. Autenticação por chave privada (PKI) — padrão corporativo
3. Como usar a classe `SnowflakeDB` com context manager
4. Funções utilitárias: `read_sql_to_dataframe` e `write_dataframe_to_snowflake`
5. Queries parametrizadas e arquivos `.sql` com Jinja2
6. Boas práticas

---

> **Credenciais:** o arquivo `rsa_key.p8` e a passphrase serão fornecidos pelo instrutor.
> Coloque a chave em `modulo5_snowflake_qualidade/chaves/rsa_key.p8`.
>
> **Segurança:** nunca commite arquivos de chave privada ou senhas no git.
> Adicione `chaves/` ao `.gitignore`.

## 1. Verificando dependências

In [ ]:
import importlib

dependencias = {
    "snowflake.connector": "snowflake-connector-python",
    "cryptography": "cryptography",
    "jinja2": "jinja2",
    "pandas": "pandas",
}

todas_ok = True
for modulo, pacote in dependencias.items():
    try:
        importlib.import_module(modulo)
        print(f"  OK  {modulo}")
    except ImportError:
        print(f"  ERRO {modulo} — instale com: uv add {pacote}")
        todas_ok = False

print()
print("Tudo pronto!" if todas_ok else "Instale as dependências acima antes de continuar.")

## 2. Importando o módulo do projeto

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório do módulo ao path do Python
modulo5_dir = Path().resolve()
if str(modulo5_dir) not in sys.path:
    sys.path.insert(0, str(modulo5_dir))

from snowflake_db import (
    SnowflakeDB,
    SNOWFLAKE_CONFIG,
    read_sql_file,
    read_sql_to_dataframe,
    write_dataframe_to_snowflake,
)

import pandas as pd

print("Módulo importado com sucesso.")
print(f"Conectará em: account={SNOWFLAKE_CONFIG['account']}")
print(f"              database={SNOWFLAKE_CONFIG['database']}")
print(f"              schema={SNOWFLAKE_CONFIG['schema']}")

## 3. Entendendo a autenticação por chave privada

A Equatorial usa **PKI (Public Key Infrastructure)** ao invés de senha para autenticar no Snowflake.
Isso é mais seguro porque:

- A chave privada nunca trafega pela rede
- Cada usuário/serviço tem sua própria chave
- É possível revogar acessos sem mudar senhas

```
Arquivo: rsa_key.p8  (chave privada — NUNCA compartilhe)
   └── Protegida por passphrase
   └── O módulo lê, descriptografa e converte para DER/PKCS8
   └── O conector Snowflake usa os bytes resultantes
```

O fluxo interno da classe `SnowflakeDB`:

In [ ]:
# Inspecionando a estrutura da classe sem executar a conexão
import inspect

# Listar os métodos públicos da classe
metodos = [
    (nome, m)
    for nome, m in inspect.getmembers(SnowflakeDB, predicate=inspect.isfunction)
    if not nome.startswith("_")
]

print("Métodos públicos de SnowflakeDB:\n")
for nome, metodo in metodos:
    sig = inspect.signature(metodo)
    doc = (metodo.__doc__ or "").strip().split("\n")[0]
    print(f"  .{nome}{sig}")
    print(f"     → {doc}")
    print()

## 4. Conectando e executando uma query

O padrão recomendado é o **context manager** (`with`).  
Ele garante que a conexão seja fechada mesmo se ocorrer um erro.

```python
with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
    df = db.query_to_dataframe("SELECT ...")
# conexão já está fechada aqui
```

In [ ]:
# Teste de conexão — verifique o contexto da sessão
try:
    with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
        df_ctx = db.query_to_dataframe(
            """
            SELECT
                CURRENT_USER()     AS usuario,
                CURRENT_ROLE()     AS role,
                CURRENT_DATABASE() AS database,
                CURRENT_SCHEMA()   AS schema,
                CURRENT_WAREHOUSE() AS warehouse,
                CURRENT_VERSION()  AS versao_snowflake
            """
        )

    print("Conexão bem-sucedida!\n")
    print(df_ctx.T.to_string(header=False))  # exibe na vertical

except Exception as e:
    print(f"Erro: {type(e).__name__}: {e}")
    print("\nVerifique:")
    print("  1. Se o arquivo rsa_key.p8 está em chaves/rsa_key.p8")
    print("  2. Se a passphrase em SNOWFLAKE_CONFIG está correta")
    print("  3. Sua conexão com a internet")

## 5. Usando `read_sql_to_dataframe`

Para análises do dia a dia, use a função utilitária — ela abre e fecha a conexão automaticamente.

In [ ]:
# Query direta como string
df_classes = read_sql_to_dataframe(
    """
    SELECT
        COD_CLASSE_CONSUMO,
        COUNT(*)                        AS QTD_UCS,
        ROUND(AVG(VLR_CONSUMO_MEDIO_KWH), 2) AS CONSUMO_MEDIO_KWH,
        SUM(CASE WHEN FLG_ATIVO THEN 1 ELSE 0 END) AS QTD_ATIVAS
    FROM CONSUMIDORES_UC
    GROUP BY COD_CLASSE_CONSUMO
    ORDER BY QTD_UCS DESC
    """
)

print(f"Linhas retornadas: {len(df_classes)}")
df_classes

## 6. Queries a partir de arquivos `.sql` com templates Jinja2

Para queries mais complexas, o ideal é salvá-las em arquivos `.sql` separados.
O módulo suporta **templates Jinja2** para passar parâmetros dinamicamente.

In [ ]:
# Criando um arquivo .sql de exemplo para demonstração
from pathlib import Path

queries_dir = Path("queries")
queries_dir.mkdir(exist_ok=True)

sql_exemplo = queries_dir / "consumidores_por_uf.sql"
sql_exemplo.write_text(
    """
SELECT
    COD_UF,
    COD_CLASSE_CONSUMO,
    COUNT(*)                             AS QTD_UCS,
    SUM(CASE WHEN CPF_CNPJ IS NULL
             THEN 1 ELSE 0 END)          AS SEM_CPF_CNPJ,
    SUM(CASE WHEN NUM_MEDIDOR IS NULL
             THEN 1 ELSE 0 END)          AS SEM_MEDIDOR
FROM CONSUMIDORES_UC
WHERE COD_UF IN ({{ ufs }})
  AND FLG_ATIVO = TRUE
GROUP BY COD_UF, COD_CLASSE_CONSUMO
ORDER BY COD_UF, QTD_UCS DESC
""",
    encoding="utf-8",
)

# Ler o arquivo processando o template
query_renderizada = read_sql_file(
    sql_exemplo,
    ufs=["MA", "PA", "GO", "PI"],  # lista vira 'MA', 'PA', 'GO', 'PI'
)

print("Query renderizada:")
print(query_renderizada)

In [ ]:
# Executar a query do arquivo
df_uf = read_sql_to_dataframe(
    sql_exemplo,
    ufs=["MA", "PA", "GO", "PI"],
)

print(f"Resultado: {len(df_uf)} linhas")
df_uf.head(10)

## 7. Queries parametrizadas (seguras contra SQL injection)

Para valores dinâmicos dentro da query, use o mecanismo de parâmetros `%s` do conector
ao invés de f-strings. Isso previne **SQL injection**.

In [ ]:
# Comparação: ERRADO vs CORRETO

cod_uf = "MA"
limite = 20

# ❌ ERRADO — vulnerável a SQL injection
# query_errada = f"SELECT * FROM CONSUMIDORES_UC WHERE COD_UF = '{cod_uf}' LIMIT {limite}"

# ✅ CORRETO — valores passados como parâmetros separados
query_segura = """
    SELECT
        COD_CONSUMIDOR,
        NOM_CONSUMIDOR,
        COD_CLASSE_CONSUMO,
        VLR_CONSUMO_MEDIO_KWH
    FROM CONSUMIDORES_UC
    WHERE COD_UF = %s
      AND FLG_ATIVO = TRUE
    ORDER BY VLR_CONSUMO_MEDIO_KWH DESC
    LIMIT %s
"""

try:
    with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
        colunas, dados = db.execute_query.__wrapped__(db, query_segura)  # sem param direto
        # Para queries parametrizadas, use o cursor diretamente:
        db.cursor.execute(query_segura, (cod_uf, limite))
        colunas = [desc[0] for desc in db.cursor.description]
        dados = db.cursor.fetchall()

    df_param = pd.DataFrame(dados, columns=colunas)
    print(f"Top {limite} consumidores de {cod_uf} por consumo médio:")
    print(df_param)

except Exception as e:
    print(f"Erro: {e}")

## 8. Gravando resultados de volta ao Snowflake

In [ ]:
import pandas as pd
from datetime import datetime

# Exemplo: criar um DataFrame com resultado de análise de qualidade
# e gravar na tabela de resultados

df_qualidade = pd.DataFrame({
    "COD_UF": ["MA", "PA", "GO"],
    "QTD_UCS": [150000, 120000, 95000],
    "PCT_SEM_CPF": [4.2, 5.8, 2.1],
    "PCT_SEM_MEDIDOR": [1.3, 2.0, 0.8],
    "SCORE_QUALIDADE": [87.5, 82.3, 93.1],
    "DAT_PROCESSAMENTO": datetime.now(),
})

print("DataFrame a ser gravado:")
print(df_qualidade)

# Para gravar de fato, descomente:
# write_dataframe_to_snowflake(
#     df_qualidade,
#     table_name="TB_RESULTADO_QUALIDADE_CADASTRO",
#     auto_create_table=True,
#     overwrite=False,
# )
# print("Dados gravados com sucesso!")

## 9. Boas práticas — resumo

| Prática | Por quê |
|---------|--------|
| Usar `with SnowflakeDB(...) as db:` | Fecha a conexão automaticamente, mesmo com erro |
| Parâmetros via `%s`, não f-string | Previne SQL injection e problemas com caracteres especiais |
| Queries em arquivos `.sql` | Facilita versionamento, revisão e reuso |
| Templates Jinja2 para listas | Gera cláusulas `IN (...)` corretamente sem concatenação manual |
| Nunca commitar `rsa_key.p8` | Chave privada comprometida = acesso ao banco comprometido |
| `LIMIT` em exploração | Snowflake cobra por dados scaneados — limite durante desenvolvimento |
| `write_pandas` com `chunk_size` | Evita timeout e erros de memória em cargas grandes |